# Session 8: センサフュージョンとカルマンフィルタ
# Session 8: Sensor Fusion and Kalman Filter

## 目標 / Objective

ジャイロ積分 vs ESKF を比較し、R パラメータの感度を分析する。

Compare gyro integration vs ESKF and analyze R parameter sensitivity.

## この Notebook で学ぶこと / What You'll Learn

| トピック | 説明 |
|---------|------|
| 相補フィルタ | シンプルなセンサフュージョン |
| ESKF | 15状態誤差状態カルマンフィルタ |
| ドリフト問題 | ジャイロ積分の限界 |
| Q/R パラメータ | プロセスノイズ vs 観測ノイズの調整 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from stampfly_edu import load_sample_data
from stampfly_edu.eskf import ESKFEducational
from stampfly_edu.eskf.wrapper import compare_gyro_vs_eskf

print("Ready! / 準備完了！")

## 1. ジャイロ積分の問題 / Gyro Integration Problem

### 理論 / Theory

ジャイロセンサは角速度 $\omega$ を計測します。角度を得るには積分が必要：

$$\theta(t) = \theta(0) + \int_0^t \omega(\tau) d\tau$$

問題：ジャイロにはバイアス $b$ とノイズ $n$ があるため：

$$\omega_{measured} = \omega_{true} + b + n$$

積分するとバイアスが時間とともに蓄積（**ドリフト**）：

$$\theta_{error} \approx b \cdot t$$

In [ ]:
# Demonstrate gyro drift
# ジャイロドリフトを実演
noise_data = load_sample_data("static_noise")

dt = noise_data['time'].iloc[1] - noise_data['time'].iloc[0]

# Integrate gyro data (stationary -> should stay at 0)
# ジャイロデータを積分（静止 → 0のはず）
roll_integrated = np.cumsum(noise_data['p'].values) * dt
pitch_integrated = np.cumsum(noise_data['q'].values) * dt
yaw_integrated = np.cumsum(noise_data['r'].values) * dt

fig, axes = plt.subplots(3, 1, figsize=(10, 8), sharex=True)

for ax, angle, label in zip(axes,
    [roll_integrated, pitch_integrated, yaw_integrated],
    ['Roll / ロール', 'Pitch / ピッチ', 'Yaw / ヨー']):
    ax.plot(noise_data['time'], np.degrees(angle), 'r-', linewidth=0.8)
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
    ax.set_ylabel(f'{label} (deg)')
    ax.grid(True, alpha=0.3)
    final = np.degrees(angle[-1])
    ax.text(0.98, 0.95, f'Final: {final:.2f}°',
            transform=ax.transAxes, ha='right', va='top',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

axes[-1].set_xlabel('Time (s) / 時間')
fig.suptitle('Gyro Integration Drift (stationary sensor) / ジャイロ積分ドリフト（静止状態）',
             fontsize=14)
fig.tight_layout()
plt.show()

print(f"After 60s of integration (should be 0°):")
print(f"  Roll:  {np.degrees(roll_integrated[-1]):.2f}°")
print(f"  Pitch: {np.degrees(pitch_integrated[-1]):.2f}°")
print(f"  Yaw:   {np.degrees(yaw_integrated[-1]):.2f}°")

## 2. 相補フィルタ / Complementary Filter

最もシンプルなセンサフュージョン：

$$\hat{\theta} = \alpha \cdot (\hat{\theta}_{prev} + \omega \cdot dt) + (1 - \alpha) \cdot \theta_{accel}$$

- ジャイロ：短期的に正確（高周波を信頼）
- 加速度計：長期的に正確（低周波を信頼）

In [ ]:
# Complementary filter implementation
# 相補フィルタの実装
def complementary_filter(gyro, accel, dt, alpha=0.98):
    """Simple complementary filter for roll/pitch."""
    n = len(gyro)
    roll_cf = np.zeros(n)
    pitch_cf = np.zeros(n)
    
    for i in range(1, n):
        # Accel-based angles
        # 加速度計ベースの角度
        roll_accel = np.arctan2(accel[i, 1], accel[i, 2])
        pitch_accel = np.arctan2(-accel[i, 0], np.sqrt(accel[i, 1]**2 + accel[i, 2]**2))
        
        # Complementary filter
        # 相補フィルタ
        roll_cf[i] = alpha * (roll_cf[i-1] + gyro[i, 0] * dt) + (1 - alpha) * roll_accel
        pitch_cf[i] = alpha * (pitch_cf[i-1] + gyro[i, 1] * dt) + (1 - alpha) * pitch_accel
    
    return roll_cf, pitch_cf

# Apply to noise data
# ノイズデータに適用
gyro = noise_data[['p', 'q', 'r']].values
accel = noise_data[['ax', 'ay', 'az']].values

# Compare different alpha values
# 異なる alpha 値を比較
fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

for alpha, color, ls in [(0.99, 'r', '-'), (0.98, 'b', '-'), (0.90, 'g', '-')]:
    roll_cf, pitch_cf = complementary_filter(gyro, accel, dt, alpha)
    axes[0].plot(noise_data['time'], np.degrees(roll_cf),
                color=color, linestyle=ls, linewidth=0.8, label=f'α={alpha}')
    axes[1].plot(noise_data['time'], np.degrees(pitch_cf),
                color=color, linestyle=ls, linewidth=0.8, label=f'α={alpha}')

# Gyro-only for comparison
axes[0].plot(noise_data['time'], np.degrees(roll_integrated),
             'k--', linewidth=0.5, alpha=0.5, label='Gyro only')

axes[0].set_ylabel('Roll (deg) / ロール')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('Pitch (deg) / ピッチ')
axes[1].set_xlabel('Time (s) / 時間')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('Complementary Filter vs Gyro-Only / 相補フィルタ vs ジャイロのみ', fontsize=14)
fig.tight_layout()
plt.show()

## 3. ESKF: 高精度センサフュージョン / ESKF: High-Precision Sensor Fusion

### ESKF の状態ベクトル（15状態）

$$\mathbf{x} = [\mathbf{p}, \mathbf{v}, \boldsymbol{\delta\theta}, \mathbf{b}_g, \mathbf{b}_a]^T$$

| 状態 | サイズ | 意味 |
|------|--------|------|
| $\mathbf{p}$ | 3 | 位置 [N, E, D] |
| $\mathbf{v}$ | 3 | 速度 |
| $\boldsymbol{\delta\theta}$ | 3 | 姿勢誤差 |
| $\mathbf{b}_g$ | 3 | ジャイロバイアス |
| $\mathbf{b}_a$ | 3 | 加速度バイアス |

In [ ]:
# Compare gyro integration vs ESKF
# ジャイロ積分 vs ESKF を比較
fig = compare_gyro_vs_eskf(noise_data, dt=dt)
plt.show()

## 4. R パラメータ感度分析 / R Parameter Sensitivity

観測ノイズ R を変更して ESKF の挙動を観察します。

In [ ]:
# R parameter sensitivity
# R パラメータ感度
from eskf import ESKFConfig

configs = [
    ("R_low (trust accel)", ESKFConfig(accel_att_noise=0.005)),
    ("R_default", ESKFConfig()),
    ("R_high (trust gyro)", ESKFConfig(accel_att_noise=0.5)),
]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

colors = ['r', 'b', 'g']
for (label, config), color in zip(configs, colors):
    eskf = ESKFEducational(config=config, enable_accel_att=True)
    
    for i in range(len(noise_data)):
        row = noise_data.iloc[i]
        g = np.array([row['p'], row['q'], row['r']])
        a = np.array([row['ax'], row['ay'], row['az']])
        eskf.predict(g, a, dt)
        eskf.update_accel_attitude(a)
        eskf.record_state(row['time'])
    
    df = eskf.get_history_df()
    axes[0].plot(df['time'], df['roll'], color=color, linewidth=0.8, label=label)
    axes[1].plot(df['time'], df['pitch'], color=color, linewidth=0.8, label=label)

axes[0].set_ylabel('Roll (deg) / ロール')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_ylabel('Pitch (deg) / ピッチ')
axes[1].set_xlabel('Time (s) / 時間')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.suptitle('ESKF R Parameter Sensitivity / ESKF R パラメータ感度', fontsize=14)
fig.tight_layout()
plt.show()

## 5. センサの有効/無効化実験 / Sensor Enable/Disable Experiment

In [ ]:
# Experiment: What happens without accelerometer correction?
# 実験: 加速度計補正なしだとどうなる？

eskf_with_accel = ESKFEducational(enable_accel_att=True)
eskf_without_accel = ESKFEducational(enable_accel_att=False)

print(f"With accel: {eskf_with_accel.sensor_status()}")
print(f"Without accel: {eskf_without_accel.sensor_status()}")

for i in range(len(noise_data)):
    row = noise_data.iloc[i]
    g = np.array([row['p'], row['q'], row['r']])
    a = np.array([row['ax'], row['ay'], row['az']])
    
    eskf_with_accel.predict(g, a, dt)
    eskf_with_accel.update_accel_attitude(a)
    eskf_with_accel.record_state(row['time'])
    
    eskf_without_accel.predict(g, a, dt)
    eskf_without_accel.update_accel_attitude(a)  # Will be ignored
    eskf_without_accel.record_state(row['time'])

df1 = eskf_with_accel.get_history_df()
df2 = eskf_without_accel.get_history_df()

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df1['time'], df1['roll'], 'b-', label='With accel correction')
ax.plot(df2['time'], df2['roll'], 'r-', label='Without accel correction')
ax.set_xlabel('Time (s) / 時間')
ax.set_ylabel('Roll (deg) / ロール')
ax.set_title('Effect of Accel Attitude Correction / 加速度計姿勢補正の効果')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. 考察課題 / Discussion Questions

1. **相補フィルタ vs ESKF**: 各手法の利点・欠点は？いつどちらを使うべきか？
2. **R パラメータの設計指針**: 大きすぎる R と小さすぎる R のそれぞれの問題は？
3. **動的環境**: ドローンが加速中（ホバーでない時）、加速度計ベースの姿勢推定はどうなるか？

---

1. **Complementary vs ESKF**: What are the pros/cons of each? When to use which?
2. **R Parameter Guidelines**: What are the problems of too-large and too-small R?
3. **Dynamic Environment**: During acceleration (non-hover), what happens to accel-based attitude?

In [ ]:
# Your experiments here / ここで実験してみてください
